In [21]:
import numpy as np
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import shap
import matplotlib.pyplot as plt

# Load the Iris dataset
iris = datasets.load_iris()
X = iris.data
y = iris.target

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train a linear SVM model
linear_svm = SVC(kernel='linear', probability=True)
linear_svm.fit(X_train, y_train)

# Make predictions
y_pred = linear_svm.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
class_report = classification_report(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)

# Print the evaluation results
print(f'Accuracy: {accuracy}')
print('Classification Report:')
print(class_report)
print('Confusion Matrix:')
print(conf_matrix)

# Explain the model's predictions using SHAP
# Create a SHAP explainer for the linear SVM
explainer = shap.Explainer(linear_svm, X_train)

# Calculate SHAP values for the test set
shap_values = explainer(X_test)
print('=====')
shap_values

Accuracy: 1.0
Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        10
           1       1.00      1.00      1.00         9
           2       1.00      1.00      1.00        11

    accuracy                           1.00        30
   macro avg       1.00      1.00      1.00        30
weighted avg       1.00      1.00      1.00        30

Confusion Matrix:
[[10  0  0]
 [ 0  9  0]
 [ 0  0 11]]
=====


.values =
array([[[-1.53753727e-02, -2.12935953e-03,  1.88466130e-01],
        [-1.42769284e-01, -4.89566542e-02, -3.32962708e-01],
        [-1.01705873e+00, -5.46456665e-01, -2.06476880e+00],
        [-1.57800175e-02, -9.91396776e-03, -5.79124496e-02]],

       [[ 3.14917272e-03,  4.36133879e-04, -3.86014966e-02],
        [ 3.78286498e-01,  1.29717266e-01,  8.82229657e-01],
        [ 1.99199076e+00,  1.07027903e+00,  4.04401464e+00],
        [ 4.01926328e-01,  2.52514591e-01,  1.47506416e+00]],

       [[-8.94735543e-02, -1.23913331e-02,  1.09673664e+00],
        [-2.46980441e-01, -8.46914383e-02, -5.76001181e-01],
        [-3.22369502e+00, -1.73206284e+00, -6.54454333e+00],
        [-5.26309996e-01, -3.30659984e-01, -1.93155052e+00]],

       [[-1.07442363e-02, -1.48798617e-03,  1.31699224e-01],
        [-9.06637061e-02, -3.10892622e-02, -2.11443472e-01],
        [-8.16455429e-01, -4.38674286e-01, -1.65751657e+00],
        [-1.55015466e-01, -9.73901539e-02, -5.68904651e-01]],

      

In [28]:
import pandas as pd
# Explain the model's predictions using SHAP
# Create a SHAP explainer for the linear SVM
explainer = shap.Explainer(linear_svm, X_train)

# Calculate SHAP values for the test set
shap_values = explainer(X_test)

# Aggregate SHAP values to get feature importance for each class
# Extract the SHAP values for each class
shap_values_per_class = shap_values.values

# Compute the mean absolute SHAP values for each feature for each class
feature_importance = np.mean(np.abs(shap_values_per_class), axis=0)

# Create a DataFrame for better visualization
feature_importance_df = pd.DataFrame(feature_importance, columns=iris.target_names, index=iris.feature_names)

print("Feature importance for each class:")
feature_importance_df

Feature importance for each class:


,setosa,versicolor,virginica
sepal length (cm),0.032616,0.004517,0.399790
sepal width (cm),0.154510,0.052983,0.360345
petal length (cm),1.658722,0.891217,3.367433
petal width (cm),0.330638,0.207727,1.213436


In [37]:
import numpy as np
from sklearn import datasets
from sklearn.model_selection import StratifiedKFold
from sklearn.svm import SVC
import shap
import pandas as pd

# Load the Iris dataset
iris = datasets.load_iris()
X = iris.data
y = iris.target

# Prepare for cross-validation
n_splits = 10
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

# Initialize an array to store feature importances for each fold
feature_importance_per_fold = np.zeros((n_splits, len(iris.feature_names), len(iris.target_names)))

# Perform 5-fold cross-validation
for fold, (train_index, test_index) in enumerate(skf.split(X, y)):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    # Train a linear SVM model
    linear_svm = SVC(kernel='linear', probability=True)
    linear_svm.fit(X_train, y_train)

    # Explain the model's predictions using SHAP
    explainer = shap.Explainer(linear_svm, X_train)
    shap_values = explainer(X_test)

    # Aggregate SHAP values to get feature importance for each class
    shap_values_per_class = shap_values.values
    feature_importance = np.mean(np.abs(shap_values_per_class), axis=0)
    feature_importance_per_fold[fold] = feature_importance

# Calculate the mean and standard deviation of feature importance across all folds
average_feature_importance = np.mean(feature_importance_per_fold, axis=0)
std_feature_importance = np.std(feature_importance_per_fold, axis=0)

# Create DataFrames for better visualization
feature_importance_df = pd.DataFrame(average_feature_importance, columns=[f"{cls} mean" for cls in iris.target_names], index=iris.feature_names)
std_feature_importance_df = pd.DataFrame(std_feature_importance, columns=[f"{cls} std" for cls in iris.target_names], index=iris.feature_names)

# Concatenate the average feature importance and standard deviation DataFrames
combined_feature_importance_df = pd.concat([feature_importance_df, std_feature_importance_df], axis=1)
# Reorder the columns to ensure the correct order: avg std avg std
combined_feature_importance_df = combined_feature_importance_df[['setosa mean',
                                                                 'setosa std',
                                                                 'versicolor mean',
                                                                 'versicolor std',
                                                                 'virginica mean',
                                                                 'virginica std']]
combined_feature_importance_df['total mean'] = combined_feature_importance_df[['setosa mean', 'versicolor mean', 'virginica mean']].sum(axis=1)
print("Average feature importance and standard deviation for each class over 5 folds:")
combined_feature_importance_df



Average feature importance and standard deviation for each class over 5 folds:


,setosa mean,setosa std,versicolor mean,versicolor std,virginica mean,virginica std,total mean
sepal length (cm),0.045268,0.042555,0.026365,0.036567,0.413651,0.171848,0.485285
sepal width (cm),0.174740,0.042219,0.059006,0.011502,0.224847,0.092296,0.458593
petal length (cm),1.500168,0.090946,0.831894,0.064708,3.068750,0.272810,5.400812
petal width (cm),0.303388,0.033255,0.182132,0.018157,1.356861,0.096603,1.842381


In [32]:
shap_values_per_class

array([[[-1.53753727e-02, -2.12935953e-03,  1.88466130e-01],
        [-1.42769284e-01, -4.89566542e-02, -3.32962708e-01],
        [-1.01705873e+00, -5.46456665e-01, -2.06476880e+00],
        [-1.57800175e-02, -9.91396776e-03, -5.79124496e-02]],

       [[ 3.14917272e-03,  4.36133879e-04, -3.86014966e-02],
        [ 3.78286498e-01,  1.29717266e-01,  8.82229657e-01],
        [ 1.99199076e+00,  1.07027903e+00,  4.04401464e+00],
        [ 4.01926328e-01,  2.52514591e-01,  1.47506416e+00]],

       [[-8.94735543e-02, -1.23913331e-02,  1.09673664e+00],
        [-2.46980441e-01, -8.46914383e-02, -5.76001181e-01],
        [-3.22369502e+00, -1.73206284e+00, -6.54454333e+00],
        [-5.26309996e-01, -3.30659984e-01, -1.93155052e+00]],

       [[-1.07442363e-02, -1.48798617e-03,  1.31699224e-01],
        [-9.06637061e-02, -3.10892622e-02, -2.11443472e-01],
        [-8.16455429e-01, -4.38674286e-01, -1.65751657e+00],
        [-1.55015466e-01, -9.73901539e-02, -5.68904651e-01]],

       [[-4.7793